## Demonstration notebook

### Day 1

- ➕ создание активного и замороженного счёта
- 🚫 попытка операций над замороженным счётом
- ✅ валидное пополнение и снятие

In [1]:
from datetime import datetime
from bank.accounts import BankAccount

day = datetime(2026, 11, 26, 15, 0)
account_active = BankAccount("12345678", "John Doe")
account_frozen = BankAccount(account_holder="Jane Smith", account_status="frozen")
account_active._clock = lambda: day
account_frozen._clock = lambda: day
print(account_active)
print(account_frozen)

BankAccount: Account holder: John Doe, Account ID: *5678, Status: active, Balance: 0.00 RUB
BankAccount: Account holder: Jane Smith, Account ID: *9171, Status: frozen, Balance: 0.00 RUB


In [2]:
try:
    account_frozen.deposit(100)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

AccountFrozenError: Operation cannot be performed on a frozen account.

In [3]:
balance = account_active.deposit(200)
print(f"Deposited 200 {account_active.currency.value}. New balance: {balance} {account_active.currency.value}.")
balance = account_active.withdraw(100)
print(f"Withdrew 100 {account_active.currency.value}. New balance: {balance} {account_active.currency.value}.")

Deposited 200 RUB. New balance: 200.00 RUB.
Withdrew 100 RUB. New balance: 100.00 RUB.


### Day 2

Создать несколько счетов SavingsAccount, PremiumAccount, InvestmentAccount и выполнить различные операции

In [1]:
from bank.accounts import InvestmentAccount, PremiumAccount, SavingsAccount

savings = SavingsAccount(account_holder="Anna", interest_rate="0.01", min_balance="50")
premium = PremiumAccount(account_holder="Boris", overdraft_limit="300", commission_fixed="15")
investment = InvestmentAccount(account_holder="Clara")
for account in (savings, premium, investment):
    account._clock = lambda: day

In [2]:
balance = savings.deposit(200)
print(f"Deposited 200 {savings.currency.value}. New balance: {balance} {savings.currency.value}.")
interest = savings.apply_monthly_interest()
print(f"Applied monthly interest: {interest} {savings.currency.value}. New balance: {savings.balance} {savings.currency.value}.")
balance = savings.withdraw(100)
print(f"Withdrew 100 {savings.currency.value}. New balance: {balance} {savings.currency.value}.")
print(savings)

Deposited 200 RUB. New balance: 200.00 RUB.
Applied monthly interest: 2.0000 RUB. New balance: 202.0000 RUB.
Withdrew 100 RUB. New balance: 102.0000 RUB.
SavingsAccount: Account holder: Anna, Account ID: *9562, Status: active, Balance: 102.0000 RUB, Interest Rate: 1.00%, Minimum Balance: 50 RUB


In [3]:
balance = premium.deposit(100)
print(f"Deposited 100 {premium.currency.value}. New balance: {balance} {premium.currency.value}.")
balance = premium.withdraw(120)
print(f"Withdrew 120 {premium.currency.value}. New balance: {balance} {premium.currency.value}.")
print(premium)

Deposited 100 RUB. New balance: 100.00 RUB.
Withdrew 120 RUB. New balance: -35.00 RUB.
PremiumAccount: Account holder: Boris, Account ID: *2730, Status: active, Balance: -35.00 RUB, Overdraft Limit: 300 RUB, Commission Fixed: 1500%


In [4]:
balance = investment.deposit(1000)
print(f"Deposited 1000 {investment.currency.value}. New balance: {balance} {investment.currency.value}.")
balance = investment.buy_funds("stock", 500)
print(f"Bought 500 {investment.currency.value} of STOCK. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.buy_funds("bond", 200)
print(f"Bought 200 {investment.currency.value} of BOND. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.buy_funds("etf", 100)
print(f"Bought 100 {investment.currency.value} of ETF. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.sell_funds("etf", 40)
print(f"Sold 40 {investment.currency.value} of ETF. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.withdraw(50)
print(f"Withdrew 50 {investment.currency.value}. New balance: {balance} {investment.currency.value}.")
growth = investment.project_yearly_growth({"stock": 0.10, "bond": 0.04, "etf": 0.07})
print(f"Projected yearly growth: {growth}.")
print(investment)

Deposited 1000 RUB. New balance: 1000.00 RUB.
Bought 500 RUB of STOCK. New portfolio: {STOCK: 500.00, BOND: 0.00, ETF: 0.00}. New balance: 500.00 RUB.
Bought 200 RUB of BOND. New portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 0.00}. New balance: 300.00 RUB.
Bought 100 RUB of ETF. New portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 100.00}. New balance: 200.00 RUB.
Sold 40 RUB of ETF. New portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 60.00}. New balance: 240.00 RUB.
Withdrew 50 RUB. New balance: 190.00 RUB.
Projected yearly growth based on provided rates: {STOCK: 50.00, BOND: 8.00, ETF: 4.20}.
InvestmentAccount: Account holder: Clara, Account ID: *0317, Status: active, Balance: 190.00 RUB, Portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 60.00}


In [5]:
try:
    savings.withdraw(1000)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

InsufficientFundsError: Withdrawal would breach minimum balance requirement. (Balance: 102.0000, Amount: 1000)

In [ ]:
try:
    premium.withdraw(500)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

InsufficientFundsError: Withdrawal would exceed overdraft limit. (Balance: -35.00, Amount: 515)

In [ ]:
try:
    investment.buy_funds("crypto", 10)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

InvalidOperationError: Unsupported investment type: 'crypto'. Expected: STOCK, BOND, ETF.

### Day 3

- клиент старше 18 лет и его счета
- заморозка счёта
- ночной запрет и пометка подозрительной операции
- три неверных пароля блокируют вход
- общий баланс и рейтинг клиентов

In [ ]:
from bank.bank import Bank

bank = Bank(clock=lambda: day)
anna = bank.add_client("Anna Ivanova", "2000-01-01", {"phone": "+79990000000"}, "secret")
boris = bank.add_client("Boris Petrov", "1990-05-05", {"email": "boris@example.com"}, "pass")
print(anna)
print(boris)

savings = bank.open_account(anna.client_id, "savings", min_balance="50")
premium = bank.open_account(boris.client_id, "premium", overdraft_limit="300", commission_fixed="15")
investment = bank.open_account(anna.client_id, "investment")
print(bank.deposit(savings.account_id, 200))
print(bank.withdraw(savings.account_id, 100))
print(bank.deposit(premium.account_id, 100))
print(bank.withdraw(premium.account_id, 120))
print(bank.deposit(investment.account_id, 1000))
print(investment.buy_funds("stock", 500))
print(bank.get_total_balance(anna.client_id))
print(bank.get_clients_ranking())

In [ ]:
bank.freeze_account(savings.account_id)
try:
    bank.deposit(savings.account_id, 10)
except Exception as exc:
    print(type(exc).__name__, exc)
bank.unfreeze_account(savings.account_id)

bank._clock = lambda: datetime(2026, 11, 26, 3, 0)
try:
    bank.deposit(premium.account_id, 10)
except Exception as exc:
    print(type(exc).__name__, exc)
print(boris.suspicious_actions)
bank._clock = lambda: day

for _ in range(3):
    try:
        bank.authenticate_client(anna.client_id, "wrong")
    except Exception as exc:
        print(type(exc).__name__, exc)
print("Locked:", anna.is_locked)